# Football Object Detection — YOLOv8s

Detect players, referees, and goalkeepers in Estonian football match images.  
Metric: mAP@[0.50:0.95]  
Data: 574 train images (516/58 split), 150 test images, all 1920×1080  
Classes: Player (0), Referee (1), Goalkeeper (2)

The YOLO dataset is already prepared in `yolo_dataset/` from the previous notebook.

In [1]:
from pathlib import Path
import csv
from ultralytics import YOLO

PROJECT_ROOT = Path(".")
YOLO_DIR = PROJECT_ROOT / "yolo_dataset"
TEST_DIR = PROJECT_ROOT / "test"
CLASS_NAMES = {0: "Player", 1: "Referee", 2: "Goalkeeper"}

## Write data.yaml

In [2]:
yaml_content = f"""path: {YOLO_DIR.resolve()}
train: images/train
val: images/val

nc: 3
names: [\"Player\", \"Referee\", \"Goalkeeper\"]
"""

with open("football.yaml", "w") as f:
    f.write(yaml_content)

print("football.yaml written")
print(f"  train: {len(list((YOLO_DIR / 'images/train').glob('*.jpg')))} images")
print(f"  val:   {len(list((YOLO_DIR / 'images/val').glob('*.jpg')))} images")

football.yaml written
  train: 516 images
  val:   58 images


## Train YOLOv8s

In [3]:
model = YOLO("yolov8s.pt")

model.train(
    data="football.yaml",
    epochs=50,
    imgsz=1280,
    batch=8,
    lr0=0.01,
    cos_lr=True,
    mosaic=1.0,
    patience=20,
    project="runs",
    name="football_yolov8s",
    device=0,
)

New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.47 🚀 Python-3.14.3 torch-2.11.0+rocm7.2 CUDA:0 (AMD Radeon Graphics, 16304MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=football.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=football_

(null): No such file or directory


Model summary: 130 layers, 11,136,761 parameters, 11,136,745 gradients, 28.7 GFLOPs

Transferred 349/355 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
ERROR ❌ AMP: checks failed. Anomalies were detected with AMP on your system that may lead to NaN losses or zero-mAP results, so AMP will be disabled during training.
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6162.6±928.6 MB/s, size: 256.5 KB)
train: Scanning /home/martin/dl4cv/dl4cv-competitions/football/yolo_dataset/labels/train.cache... 516 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 516/516 196.8Mit/s 0.0s
train: /home/martin/dl4cv/dl4cv-competitions/football/yolo_dataset/images/train/0355_nzAbA9Nk03I_shot2.jpg: 1 duplicate labels removed
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4556.6±625.3 MB/s, size: 290.9 KB)
val: Scanning /home/martin/dl4cv/dl4cv-competitions/football/yolo_dataset/labels/val.cache... 58 images, 0 background

Exception in thread Thread-6 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/martin/.local/share/uv/python/cpython-3.14.3-linux-x86_64-gnu/lib/python3.14/threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/home/martin/.local/share/uv/python/cpython-3.14.3-linux-x86_64-gnu/lib/python3.14/threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/martin/dl4cv/dl4cv-competitions/.venv/lib/python3.14/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
    ~~~~~~~~~~~^^
  File "/home/martin/dl4cv/dl4cv-competitions/.venv/lib/python3.14/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/home/martin/.local/share/uv/python/cpython-3.14.3-linux-x86_64-gnu/lib/python3.14/multiprocessing/queues

KeyboardInterrupt: 

## Evaluate on validation set

In [4]:
best_model = YOLO("runs/detect/runs/football_yolov8s/weights/best.pt")
metrics = best_model.val(data="football.yaml", imgsz=1280, device=0)
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

Ultralytics 8.4.47 🚀 Python-3.14.3 torch-2.11.0+rocm7.2 CUDA:0 (AMD Radeon Graphics, 16304MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5497.2±853.8 MB/s, size: 269.5 KB)
val: Scanning /home/martin/dl4cv/dl4cv-competitions/football/yolo_dataset/labels/val.cache... 58 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 58/58 34.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.0it/s 4.0s1.6s
                   all         58        697      0.785      0.726      0.804      0.462
                Player         57        602      0.958      0.872      0.952      0.544
               Referee         41         58      0.741      0.793      0.856      0.484
            Goalkeeper         36         37      0.657      0.514      0.604      0.357
Speed: 0.4ms preprocess, 58.5ms inference, 0.0ms loss, 0.4ms postprocess per image


## Generate submission

In [5]:
best_model = YOLO("runs/detect/runs/football_yolov8s/weights/best.pt")

rows = []
for img_path in sorted(TEST_DIR.glob("*.jpg")):
    result = best_model.predict(
        source=str(img_path),
        imgsz=1280,
        conf=0.25,
        iou=0.45,
        augment=True,   # TTA
        verbose=False,
    )[0]

    preds = []
    if result.boxes is not None:
        for box in result.boxes:
            cls_name = CLASS_NAMES[int(box.cls.item())]
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            preds.append(f"{cls_name} {x1} {y1} {x2} {y2}")

    rows.append((img_path.name, " ".join(preds) if preds else "Player 960 540 1000 600"))

with open("submission_yolov8s.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image_name", "predictions"])
    writer.writerows(rows)

print(f"submission_yolov8s.csv written — {len(rows)} images")

import pandas as pd
df = pd.read_csv("submission_yolov8s.csv")
df.head(3)

submission_yolov8s.csv written — 150 images


,image_name,predictions
0,0003_Th3oNOuYC80_shot1.jpg,Player 794 467 899 657 Referee 1825 236 1886 3...
1,0003_Th3oNOuYC80_shot3.jpg,Player 289 311 383 474 Player 775 291 877 485 ...
2,0005_acnnLx26C7E_shot1.jpg,Referee 1686 455 1733 562 Player 742 421 791 5...


## (Optional) Fine-tune on train+val, re-generate submission

After picking the best config, retrain on the full dataset for the final submission.

In [ ]:
import xml.etree.ElementTree as ET
import shutil

TRAIN_IMG_DIR = PROJECT_ROOT / "train" / "images"
TRAIN_ANN_DIR = PROJECT_ROOT / "train" / "annotations"
FULL_DIR = PROJECT_ROOT / "yolo_dataset_full"

CLASS_MAP = {"Player": 0, "Referee": 1, "Goalkeeper": 2}
IMG_W, IMG_H = 1920, 1080

(FULL_DIR / "images" / "train").mkdir(parents=True, exist_ok=True)
(FULL_DIR / "labels" / "train").mkdir(parents=True, exist_ok=True)

def voc_to_yolo(xmin, ymin, xmax, ymax):
    cx = ((xmin + xmax) / 2) / IMG_W
    cy = ((ymin + ymax) / 2) / IMG_H
    w  = (xmax - xmin) / IMG_W
    h  = (ymax - ymin) / IMG_H
    return cx, cy, w, h

all_xmls = sorted(TRAIN_ANN_DIR.glob("*.xml"))
for xml_path in all_xmls:
    stem = xml_path.stem
    for ext in [".jpg", ".jpeg", ".png"]:
        src = TRAIN_IMG_DIR / (stem + ext)
        if src.exists():
            shutil.copy(src, FULL_DIR / "images" / "train" / src.name)
            break
    labels = []
    for obj in ET.parse(xml_path).iter("object"):
        cls = obj.find("name").text
        if cls not in CLASS_MAP:
            continue
        bb = obj.find("bndbox")
        cx, cy, w, h = voc_to_yolo(
            int(bb.find("xmin").text), int(bb.find("ymin").text),
            int(bb.find("xmax").text), int(bb.find("ymax").text)
        )
        labels.append(f"{CLASS_MAP[cls]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    with open(FULL_DIR / "labels" / "train" / f"{stem}.txt", "w") as f:
        f.write("\n".join(labels))

print(f"Full dataset: {len(all_xmls)} images")

full_yaml = f"""path: {FULL_DIR.resolve()}
train: images/train
val: images/train

nc: 3
names: [\"Player\", \"Referee\", \"Goalkeeper\"]
"""
with open("football_full.yaml", "w") as f:
    f.write(full_yaml)
print("football_full.yaml written")

In [ ]:
full_model = YOLO("runs/detect/football_yolov8s/weights/best.pt")

full_model.train(
    data="football_full.yaml",
    epochs=10,
    imgsz=1280,
    batch=8,
    lr0=0.001,
    cos_lr=True,
    patience=0,
    project="runs",
    name="football_yolov8s_full",
    device=0,
)

In [ ]:
final_model = YOLO("runs/detect/football_yolov8s_full/weights/last.pt")

rows = []
for img_path in sorted(TEST_DIR.glob("*.jpg")):
    result = final_model.predict(
        source=str(img_path),
        imgsz=1280,
        conf=0.25,
        iou=0.45,
        augment=True,
        verbose=False,
    )[0]

    preds = []
    if result.boxes is not None:
        for box in result.boxes:
            cls_name = CLASS_NAMES[int(box.cls.item())]
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            preds.append(f"{cls_name} {x1} {y1} {x2} {y2}")

    rows.append((img_path.name, " ".join(preds) if preds else "Player 960 540 1000 600"))

with open("submission_yolov8s_full.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image_name", "predictions"])
    writer.writerows(rows)

print(f"submission_yolov8s_full.csv written — {len(rows)} images")